In [ ]:
from pathlib import Path
import os
import sys

import mne
import numpy as np
project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)

os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%matplotlib qt

In [ ]:
# Baixa automaticamente arquivos ausentes; reutiliza o fsaverage salvo nas proximas execucoes.
subjects_dir = project_root / "data" / "models"
subjects_dir.mkdir(parents=True, exist_ok=True)
fs_dir = mne.datasets.fetch_fsaverage(subjects_dir=subjects_dir, verbose=True)

In [ ]:
from utils.decode_trigger import decode_8bit_trigger, convert_dict_trigger
from modules.events import get_events_tms_per_task
from modules.preprocessing import fix_stim_artifact_cubic
from modules.prime.sound import sound
from modules.prime.ssp_sir import ssp_sir_to_average, ssp_sir_trials
from utils.validation_data import iterative_zscore_rejection

In [ ]:
# Inclua os arquivos na ordem de aquisicao.
files = ["data/raw/V1.bdf"]
raws = [mne.io.read_raw_bdf(file, preload=True) for file in files]
raw = mne.concatenate_raws(raws)

In [ ]:
raw_data = raw.copy()

In [ ]:
emg_ch_names = ["EMG L", "EMG R"]
eog_ch_names = ["EOG"]
raw_data.set_channel_types({ch: "emg" for ch in emg_ch_names})
raw_data.set_channel_types({'EOG':'eog'})

montage = mne.channels.make_standard_montage("standard_1020",head_size='auto')
raw_data.set_montage(montage)

In [ ]:
events, event_id = mne.events_from_annotations(raw_data)
event_id, renamed_dict =  convert_dict_trigger(event_id, decode_8bit_trigger)
raw_data.annotations.rename(renamed_dict)
raw_data.event_id = event_id

In [ ]:
raw_data.plot(scalings={"eeg": 100e-6, "emg": 500e-6, "eog": 200e-6}, n_channels=10)

In [ ]:
bad_ch =[]
raw_data.drop_channels(bad_ch)

In [ ]:
events_tms, events_id_tms = get_events_tms_per_task(events, event_id)

In [ ]:
raw_data = fix_stim_artifact_cubic(
    inst=raw_data, 
    events=events,
    event_id=event_id["tms_pulse"],
    tmin=-0.005,
    tmax=0.010,
    pre_window=0.010,
    post_window=0.010
)

In [ ]:
epochs = mne.Epochs(
    raw_data,
    events=events_tms,
    event_id=events_id_tms,
    tmin=-0.8,
    tmax=0.8,
    baseline=None,
    preload=True,
)

In [ ]:
epochs.average().plot()

In [ ]:
epochs.plot(block=True)

In [ ]:
# A amplitude da resposta TMS nao entra na triagem automatica.
pre_data = epochs.copy().crop(tmin=-0.5, tmax=-0.01).get_data(picks="eeg", copy=True)
_, kept_indices, rejected_indices = iterative_zscore_rejection(pre_data, threshold=2.5)
epochs.drop(rejected_indices.tolist(), reason="pre_stim_zscore")

In [ ]:
indices_removidos = [i for i, log in enumerate(epochs.drop_log) if log]

print(f"Foram removidas {len(indices_removidos)} épocas.")
print("Índices:", indices_removidos)

In [ ]:
bad_epochs = []
epochs.drop(bad_epochs)

In [ ]:
epochs.set_eeg_reference("average", projection=False)
epochs_ica = epochs.copy().crop(tmin=None, tmax=-0.01)
epochs_ica.filter(
    l_freq=1, h_freq=100, picks="eeg", method="iir",
    iir_params=dict(order=2, ftype="butter"),
)
ica = mne.preprocessing.ICA(
    n_components=0.995, random_state=97, max_iter=800,
    method="infomax", fit_params=dict(extended=True),
)
ica.fit(epochs_ica)

In [ ]:
ica.plot_sources(epochs, show_scrollbars=True)

In [ ]:
ica.exclude = []
epochs_clean = ica.apply(epochs.copy())

In [ ]:
epochs_clean.apply_baseline(baseline=(-0.50, -0.01))

In [ ]:
# Forward fsaverage calculado para os canais EEG atuais.
eeg_picks = mne.pick_types(epochs_clean.info, eeg=True, meg=False, exclude="bads")
eeg_info = mne.pick_info(epochs_clean.info, eeg_picks)
eeg_names = eeg_info["ch_names"]

source_space = fs_dir / "bem" / "fsaverage-ico-5-src.fif"
bem = fs_dir / "bem" / "fsaverage-5120-5120-5120-bem-sol.fif"
forward = mne.make_forward_solution(
    eeg_info, trans="fsaverage", src=source_space, bem=bem,
    eeg=True, meg=False, mindist=5.0, n_jobs=1, verbose=False,
)
assert forward["sol"]["row_names"] == eeg_names
leadfield = forward["sol"]["data"]
leadfield = leadfield - leadfield.mean(axis=0, keepdims=True)

In [ ]:
# Estima SOUND no pos-estimulo e aplica a mesma matriz a todas as epocas EEG.
post_mask = epochs_clean.times > 0
post_average = epochs_clean.get_data(picks=eeg_names, copy=True).mean(axis=0)[:, post_mask]
sound_filter, sound_sigmas, sound_iterations, sound_converged = sound(
    post_average.T, baseline_correction=0,
    sigmas=np.ones(len(eeg_names)), num_of_channels=len(eeg_names),
    lfm=leadfield, iterations=10, lambda0=0.01, convergence_boundary=1e-9,
)
clean_data = epochs_clean.get_data(copy=False)
clean_data[:, eeg_picks, :] = sound_filter @ clean_data[:, eeg_picks, :]

In [ ]:
# A janela automatica de estimacao do artefato tem 50 milissegundos.
post_average = epochs_clean.get_data(picks=eeg_names, copy=True).mean(axis=0)[:, post_mask]
(_, artifact_topographies, _, post_kernel, suppression_matrix, sir_rank, _,
 sir_suppressed, sir_original, n_pcs_removed) = ssp_sir_to_average(
    post_average, leadfield, epochs_clean.info["sfreq"],
    timerange=("automatic", 50), method=("threshold", 0.99),
)
filter_kernel = np.zeros((len(eeg_names), len(epochs_clean.times)))
filter_kernel[:, post_mask] = post_kernel
clean_data[:, eeg_picks, :] = ssp_sir_trials(
    clean_data[:, eeg_picks, :], P=suppression_matrix,
    sir_projmat_suppr=sir_suppressed, sir_projmat_orig=sir_original,
    filter_kernel=filter_kernel,
)

In [ ]:
epochs_clean = fix_stim_artifact_cubic(
    inst=epochs_clean, 
    tmin=-0.005,
    tmax=0.010,
    pre_window=0.010,
    post_window=0.010
)

In [ ]:
# O passa-baixa final vem depois da estimacao de artefato acima de 100 Hz.
epochs_clean.filter(l_freq=None, h_freq=45, fir_design="firwin")
epochs_clean.apply_baseline(baseline=(-0.5, -0.01))

In [ ]:
epochs_clean.copy().average().plot()
# Revise artefatos residuais pos-TMS; amplitudes altas nao sao rejeitadas automaticamente.
epochs_clean.plot(block=True)
print({condition: len(epochs_clean[condition]) for condition in epochs_clean.event_id})

In [ ]:
output_path = Path("data/processed/epochs-tep.fif")
output_path.parent.mkdir(parents=True, exist_ok=True)
epochs_clean.save(output_path, overwrite=True)
ica.save(output_path.with_name("tep-ica.fif"), overwrite=True)
np.savez(
    output_path.with_name("tep-cleaning.npz"),
    forward_model="fsaverage",
    eeg_names=np.array(eeg_names), sound_filter=sound_filter,
    sound_sigmas=sound_sigmas, sound_iterations=sound_iterations,
    sound_converged=sound_converged, sound_lambda=0.01,
    sspsir_window_ms=50, sspsir_threshold=0.99,
    sspsir_pcs_removed=n_pcs_removed, sspsir_rank=sir_rank,
    sspsir_suppression=suppression_matrix, sspsir_kernel=filter_kernel,
    sspsir_sir_suppressed=sir_suppressed, sspsir_sir_original=sir_original,
)